# Unit 2, Lecture 2: Tool calling, in depth

In Unit 1 every tool returned clean data on the first try. That was a teaching
convenience. Real tools time out, throw, hit rate limits, and worst of all
**succeed while returning garbage**. This notebook breaks a tool four ways and
builds one function that survives all of them.

The example is a weather assistant, chosen because the data is trivial, so
nothing distracts from the one thing we are studying: what happens when the tool
itself misbehaves.

## Setup

In [1]:
import sys

from pathlib import Path
project_root = Path.cwd().parents[1]
sys.path.insert(0, str(project_root / "src"))

In [2]:
from cse476.lanes import get_client, MODEL, describe
from cse476.tools import (
    call_tool, run_with_tools,
    get_weather,
    get_weather_that_throws,
    get_weather_that_returns_junk,
    get_weather_that_rate_limits,
)

print(describe())
client = get_client()
print("happy path:", get_weather("Mumbai"))

Lane: Groq (groq, free)  |  Model: openai/gpt-oss-120b
happy path: Mumbai: 31C, humid, light rain.


## 1. The four ways a tool fails, raw and unprotected

Run each broken tool directly, with no defence, so you can see exactly how it
misbehaves. **The junk one is the dangerous one**, because nothing errors.

In [3]:
# throws
try:
    get_weather_that_throws("Delhi")
except Exception as e:
    print("THROWS   ->", type(e).__name__, str(e))

# returns junk: no exception at all, just useless output
junk = get_weather_that_returns_junk("Delhi")
print("JUNK     ->", repr(junk))
print("           notice: nothing errored. This would flow straight to the model.")

THROWS   -> ToolError Connection reset by peer while reaching the weather service.
JUNK     -> '\x00\x00 <html>502 Bad Gateway</html> \x00'
           notice: nothing errored. This would flow straight to the model.


Look hard at the junk case. There is no exception, no warning, nothing in a
log. An error page came back dressed as data. Without a check, the model reads
`<html>502 Bad Gateway</html>` as weather and tells your user it is 502 degrees.
That is the failure that causes real incidents, because it is silent.

## 2. The defended executor

`call_tool` wraps any tool with four defences: a whitelist, retry with backoff,
a catch, and a usability check. Whatever happens, it returns a `ToolResult` with
a **readable observation** the model can reason about. It never raises.

In [4]:
# happy path
print(call_tool("get_weather", {"city": "Mumbai"}))
print()

# an unknown tool never runs
print(call_tool("delete_everything", {}))

ToolResult(name='get_weather', ok=True, observation='Mumbai: 31C, humid, light rain.', attempts=1, seconds=0.0)

ToolResult(name='delete_everything', ok=False, observation="Error: no tool named 'delete_everything'. Available: ['get_weather']. Use one of those or answer without a tool.", attempts=1, seconds=0.0)


In [5]:
# the junk tool: caught and turned into an honest observation
r = call_tool("get_weather", {"city": "Delhi"},
              registry={"get_weather": get_weather_that_returns_junk})
print("ok:         ", r.ok)
print("observation:", r.observation)
print("null bytes passed on?", "\x00" in r.observation)

ok:          False
observation: Tool 'get_weather' returned unusable output. Treat it as no data and either try a different approach or say so.
null bytes passed on? False


## 3. Permanent versus transient

Not every failure should be retried. A dropped connection might work next time,
so retry it. **Wrong arguments will fail identically forever**, so retrying just
wastes calls. `call_tool` catches a `TypeError` (bad arguments) and returns at
once, while treating everything else as possibly transient.

In [6]:
# permanent: bad arguments, tried exactly once
def needs_city(city: str) -> str:
    return get_weather(city)

r = call_tool("get_weather", {"wrong_arg": "x"},
              registry={"get_weather": needs_city}, retries=2, backoff=0.0)
print(f"permanent fault -> attempts: {r.attempts}  (retrying would be pointless)")

# transient: fails once, then recovers
calls = {"n": 0}
def flaky(city: str) -> str:
    calls["n"] += 1
    if calls["n"] < 2:
        raise RuntimeError("temporary blip")
    return get_weather(city)

r = call_tool("get_weather", {"city": "Jammu"},
              registry={"get_weather": flaky}, retries=2, backoff=0.0)
print(f"transient fault -> attempts: {r.attempts}  ok: {r.ok}  ({r.observation})")

permanent fault -> attempts: 1  (retrying would be pointless)
transient fault -> attempts: 2  ok: True  (Jammu: 29C, clear.)


## 4. The loop, hardened

Now put the defended executor inside the agent loop. Watch the model recover
from a tool that fails every time: the loop does not crash, the failure goes
back as a readable sentence, and the model does the honest thing.

In [7]:
result = run_with_tools(
    client, MODEL,
    "What is the weather in Delhi?",
    registry={"get_weather": get_weather_that_throws},
)
print()
print("final answer:", result.answer)
print("stopped because:", result.stopped_because)

  [step 1] get_weather({'city': 'Delhi'}) [FAILED] -> Tool 'get_weather' failed after 3 attempts. Last error: Connection res

final answer: I’m sorry—I wasn’t able to retrieve the weather data for Delhi. The weather service didn’t respond, so I don’t have a current forecast to share. If you need up‑to‑date weather information, you might try checking a reliable weather website or app.
stopped because: goal met


The tool failed three times. The loop survived. The model, told in its system
prompt not to pretend when a tool fails, told the user the truth instead of
inventing a temperature. **That honesty was made possible by how we handled the
failure, not by anything clever the model did.**

## Your turn

**1. Break your own tool.** Take a tool from your Practical 2 agent. Write three
broken versions: one that throws, one that returns an empty string, one that
returns junk. Run each through `call_tool` and confirm none of them crashes.

**2. Add a fifth failure mode.** The four here are not the only ones. Invent a
fifth realistic failure, add a defence for it to `call_tool` (or to
`_looks_usable`), and say in one sentence whether it is transient or permanent.

**3. Write the honest sentence.** When a tool fails and your agent cannot answer,
what exactly should it say to the user? Write that sentence. A vague failure
message is its own kind of bug.

In [8]:
# your work here
